# 01 — Scikit-Learn Basics (Mastery Series)

Welcome to Part 1 of the **Scikit-Learn Mastery Series**!

### What this notebook covers:
1. **Introduction & Setup**: Core Estimator API design (`fit`, `predict`, `transform`)
2. **Dataset Management**: Built-in toy datasets & synthetic dataset generators
3. **Exploratory Data Analysis**: Pandas DataFrames & feature visualization
4. **Train/Test Splitting**: Stratified splits to preserve class distributions
5. **Feature Scaling**: `StandardScaler`, `MinMaxScaler`, `RobustScaler`
6. **Categorical Encoding & Imputation**: `OneHotEncoder`, `OrdinalEncoder`, `SimpleImputer`
7. **ColumnTransformer**: Preprocessing heterogenous numeric & categorical data
8. **Pipelines**: Chaining preprocessing + models & preventing data leakage
9. **Model Evaluation Metrics**: Classification & Regression evaluation metrics
10. **Cross-Validation, Tuning & Persistence**: `GridSearchCV`, `joblib`, and Scikit-Learn Best Practices Cheat Sheet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn import datasets

print(f'Scikit-Learn Version: {sklearn.__version__}')
print('NumPy Version:', np.__version__)
print('Pandas Version:', pd.__version__)

## 2. Loading & Generating Datasets

Scikit-Learn provides built-in toy datasets (`load_*`) and synthetic data generators (`make_*`).

In [ ]:
# 1. Loading built-in toy classification dataset
iris = datasets.load_iris()
print('--- Iris Classification Dataset ---')
print('Feature Names:', iris.feature_names)
print('Target Names:', iris.target_names)
print('Data Shape:', iris.data.shape)

# 2. Loading built-in toy regression dataset
diabetes = datasets.load_diabetes()
print('\n--- Diabetes Regression Dataset ---')
print('Feature Names:', diabetes.feature_names)
print('Data Shape:', diabetes.data.shape)

In [ ]:
from sklearn.datasets import make_classification, make_regression, make_blobs

# Generating synthetic classification dataset
X_syn_cls, y_syn_cls = make_classification(
    n_samples=200, n_features=4, n_informative=3, n_redundant=1, random_state=42
)
print('Synthetic Classification Shape:', X_syn_cls.shape, y_syn_cls.shape)

# Generating synthetic regression dataset
X_syn_reg, y_syn_reg = make_regression(
    n_samples=150, n_features=3, noise=10.0, random_state=42
)
print('Synthetic Regression Shape:', X_syn_reg.shape, y_syn_reg.shape)

## 3. Exploratory Data Analysis (EDA) & DataFrame Integration

Converting Scikit-learn Bunch objects into Pandas DataFrames simplifies exploration, summary stats, and visualization.

In [ ]:
iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df['target'] = iris.target
iris_df['species'] = iris_df['target'].map({i: name for i, name in enumerate(iris.target_names)})

print('--- DataFrame Head ---')
print(iris_df.head())

print('\n--- Summary Statistics ---')
print(iris_df.describe().round(2))

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.scatterplot(data=iris_df, x='petal length (cm)', y='petal width (cm)', hue='species', palette='viridis')
plt.title('Petal Length vs Petal Width')

plt.subplot(1, 2, 2)
sns.heatmap(iris_df.iloc[:, :4].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. The Estimator API & Train/Test Splitting

Every model in Scikit-Learn adheres to a unified API pattern:
- `fit(X, y)`: Train model / compute parameters
- `predict(X)`: Make predictions on new samples
- `transform(X)`: Apply transformation (for scalers & encoders)
- `fit_transform(X, y)`: Fit parameters then transform in one step

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = iris.data
y = iris.target

# Stratified train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set shape: {X_train.shape}, Test set shape: {X_test.shape}')
print('Target class distribution in train:', np.bincount(y_train))
print('Target class distribution in test :', np.bincount(y_test))

In [ ]:
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('First 5 Test Predictions:', y_pred[:5])
print('First 5 Actual Labels    :', y_test[:5])

## 5. Feature Scaling Transformers

Machine learning algorithms (especially distance-based and gradient-based algorithms) benefit significantly from feature scaling.

- **StandardScaler**: Standardizes features by removing the mean and scaling to unit variance (`z = (x - u) / s`).
- **MinMaxScaler**: Scales features to a bounded range, typically `[0, 1]` (`z = (x - min) / (max - min)`).
- **RobustScaler**: Scales features using statistics that are robust to outliers (median and Interquartile Range).

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Initialize scalers
std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()
robust_scaler = RobustScaler()

# Fit scaler ONLY on training data to avoid data leakage
X_train_std = std_scaler.fit_transform(X_train)
X_test_std = std_scaler.transform(X_test)

X_train_minmax = minmax_scaler.fit_transform(X_train)
X_train_robust = robust_scaler.fit_transform(X_train)

print('Original Mean & Std (Feature 0):', X_train[:, 0].mean().round(2), X_train[:, 0].std().round(2))
print('StandardScaler Mean & Std      :', X_train_std[:, 0].mean().round(2), X_train_std[:, 0].std().round(2))
print('MinMaxScaler Min & Max         :', X_train_minmax[:, 0].min(), X_train_minmax[:, 0].max())

## 6. Categorical Encoding & Missing Value Imputation

Real-world data often includes missing values and non-numeric categorical features.
- `LabelEncoder`: Encodes target labels into sequential integers (`0, 1, 2`).
- `OneHotEncoder`: Converts nominal categorical variables into binary indicator vectors.
- `OrdinalEncoder`: Encodes ordinal categorical features into ordered numbers.
- `SimpleImputer`: Fills missing values using mean, median, mode, or constant.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

# 1. Missing Value Imputation
data_with_nans = np.array([[1.0, 2.0, np.nan], [3.0, np.nan, 6.0], [7.0, 8.0, 9.0]])
imputer = SimpleImputer(strategy='mean')
imputed_data = imputer.fit_transform(data_with_nans)
print('--- Imputed Data (Mean Strategy) ---\n', imputed_data.round(2))

# 2. One-Hot Encoding
cats = np.array([['Red'], ['Blue'], ['Green'], ['Blue']])
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
onehot_encoded = ohe.fit_transform(cats)
print('\n--- One-Hot Encoded Features ---\n', onehot_encoded)
print('Categories:', ohe.get_feature_names_out(['color']))